# Native hadamard workflow
Native local quantum program, synthesis, CPU execution and named readout. Seeded host samples are drawn from exact local probabilities; no remote sampling claim. Run every cell from the repository root; see WORKFLOW.md for scope and adaptations. Retain the original 100000 shots. Dropped environment remains entangled and is not clean workspace.


In [ ]:
import math
from collections import Counter
from random import Random

import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import (
    H, Output, QArray, QBit, allocate, apply, control, drop, qft, qfunc, within,
)


In [ ]:
@qfunc
def controlled_qft(expectation_value: QBit, psi: QArray[QBit]) -> None:
    control(expectation_value, lambda: qft(psi))

@qfunc
def preparation_and_application(expectation_value: QBit, psi: QArray[QBit]) -> None:
    with within():
        H(expectation_value)
    with apply():
        controlled_qft(expectation_value, psi)

@qfunc
def main(expectation_value: Output[QBit]) -> None:
    psi: QArray[QBit] = QArray()
    allocate(expectation_value)
    allocate(4, psi)
    preparation_and_application(expectation_value, psi)
    drop(psi)


In [ ]:
program = synth.synthesize(main, max_workspace=0)
result = fq.run(program)
probabilities = dict(program.probabilities(result, registers=("expectation_value",)))
NUM_SHOTS = 100_000
counts = Counter(Random(17).choices(list(probabilities), weights=list(probabilities.values()), k=NUM_SHOTS))
P_0 = counts[(0,)] / NUM_SHOTS
P_1 = counts[(1,)] / NUM_SHOTS
expectation_value = 2 * P_0 - 1
print("P_0:", P_0, "P_1:", P_1)
print("Re(<U>):", expectation_value)
print("Exact Re(<U>):", 0.25)
assert abs(expectation_value - 0.25) < 0.02
print(program.report["resources"])
